In [55]:
import sys
import pandas as pd
import numpy as np

from measurements import *
from data_utils import *


In [45]:
file_path = '/n/data1/hms/dbmi/manrai/aashna/NORMA/data/raw/mimiciv/labevents.csv'
df = pd.read_csv(file_path, usecols=['subject_id', 'itemid', 'valuenum', 'valueuom'])
display(df.head(10))

process_type = "ROUTINE"
labid_col = "itemid"
label_col = "label"
labval_col = "valuenum"
labunit_col = "valueuom"
    
# df = df[df.priority == process_type].query('itemid == 52142')
# df

,subject_id,itemid,valuenum,valueuom
0,10000032,50931,95.0,mg/dL
1,10000032,51071,NaN,NaN
2,10000032,51074,NaN,NaN
3,10000032,51075,NaN,NaN
4,10000032,51079,NaN,NaN
5,10000032,51087,NaN,NaN
6,10000032,51089,NaN,NaN
7,10000032,51090,NaN,NaN
8,10000032,51092,NaN,NaN
9,10000032,50853,15.0,ng/mL


In [49]:
df_mpv = df.query('itemid == 51074')
df_mpv


,subject_id,itemid,valuenum,valueuom
2,10000032,51074,NaN,NaN
123,10000032,51074,NaN,NaN
207,10000032,51074,NaN,NaN
760,10000084,51074,NaN,NaN
961,10000084,51074,NaN,NaN
...,...,...,...,...
158362996,19999466,51074,NaN,NaN
158363825,19999784,51074,NaN,NaN
158373915,19999829,51074,NaN,NaN
158373981,19999840,51074,NaN,NaN


In [52]:
data_dir = '/n/data1/hms/dbmi/manrai/aashna/NORMA/data/raw/mimiciv/3.1/'
meta_df = pd.read_csv(data_dir + "hosp/d_labitems.csv")
str_lbs = [x for x in meta_df.label.unique() if isinstance(x, str)]
meta_df


,itemid,label,fluid,category
0,50801,Alveolar-arterial Gradient,Blood,Blood Gas
1,50802,Base Excess,Blood,Blood Gas
2,50803,"Calculated Bicarbonate, Whole Blood",Blood,Blood Gas
3,50804,Calculated Total CO2,Blood,Blood Gas
4,50805,Carboxyhemoglobin,Blood,Blood Gas
...,...,...,...,...
1645,53186,MCH,Blood,Chemistry
1646,53187,PAN,Stool,Chemistry
1647,53188,Lymphocytes,Blood,Chemistry
1648,53189,Platelet Count,Blood,Chemistry


In [13]:
file_path = '/n/data1/hms/dbmi/manrai/aashna/NORMA/data/raw/mimiciv/labevents.csv'
df = pd.read_csv(file_path, nrows=1000) 
df.head(100)


,labevent_id,subject_id,hadm_id,specimen_id,itemid,order_provider_id,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,priority,comments
0,1,10000032,NaN,2704548,50931,P69FQC,2180-03-23 11:51:00,2180-03-23 15:56:00,___,95.0,mg/dL,70.0,100.0,NaN,ROUTINE,"IF FASTING, 70-100 NORMAL, >125 PROVISIONAL DI..."
1,2,10000032,NaN,36092842,51071,P69FQC,2180-03-23 11:51:00,2180-03-23 16:00:00,NEG,NaN,NaN,NaN,NaN,NaN,ROUTINE,NaN
2,3,10000032,NaN,36092842,51074,P69FQC,2180-03-23 11:51:00,2180-03-23 16:00:00,NEG,NaN,NaN,NaN,NaN,NaN,ROUTINE,NaN
3,4,10000032,NaN,36092842,51075,P69FQC,2180-03-23 11:51:00,2180-03-23 16:00:00,NEG,NaN,NaN,NaN,NaN,NaN,ROUTINE,BENZODIAZEPINE IMMUNOASSAY SCREEN DOES NOT DET...
4,5,10000032,NaN,36092842,51079,P69FQC,2180-03-23 11:51:00,2180-03-23 16:00:00,NEG,NaN,NaN,NaN,NaN,NaN,ROUTINE,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,10000032,NaN,95700408,50861,NaN,2180-05-06 22:25:00,2180-05-06 23:16:00,100,100.0,IU/L,0.0,40.0,abnormal,STAT,NaN
96,97,10000032,NaN,95700408,50862,NaN,2180-05-06 22:25:00,2180-05-06 23:16:00,3.3,3.3,g/dL,3.5,5.2,abnormal,STAT,NaN
97,98,10000032,NaN,95700408,50863,NaN,2180-05-06 22:25:00,2180-05-06 23:16:00,114,114.0,IU/L,35.0,105.0,abnormal,STAT,NaN
98,99,10000032,NaN,95700408,50868,NaN,2180-05-06 22:25:00,2180-05-06 23:16:00,9,9.0,mEq/L,8.0,20.0,NaN,STAT,NaN


In [56]:
def get_pretty_lab_stats_df():
    """
    Reads combined_lab_stats_by_test_v2.csv and returns a pretty DataFrame
    pivoted by source with multi-index columns:
    ('num_seq', 'mean', 'bounds', 'count_mean').
    Handles files that may or may not have value_count/value_count_std columns.
    Fixes handling of missing means and correct column names/order.
    """
    lab_stats = pd.read_csv('../data/processed/combined_lab_stats_by_test_v2.csv')
    lab_stats['test_name'] = lab_stats['test_name'].fillna('NA')
    lab_stats['source'] = lab_stats['source'].replace('mimiciv', 'MIMIC-IV')
    lab_stats['source'] = lab_stats['source'].replace('ehrshot', 'EHRSHOT')
    keep_columns = ['source', 'test_name', 'n_sequences', 'value_mean', 'value_std', 'value_min', 'value_max', 'count_mean', 'count_std']
    lab_stats = lab_stats[[c for c in keep_columns if c in lab_stats.columns]]

    def pretty_mean(row):
        return f"{row['value_mean']:.2f} ± {row['value_std']:.2f}"

    def pretty_bounds(row):
        return f"[{row['value_min']:.2f}, {row['value_max']:.2f}]"

    def pretty_count_mean(row):
        return f"{row['count_mean']:.2f} ± {row['count_std']:.2f}"
    
    # Set human-friendly display column keys matching newly added columns to lab_stats
    display_cols = [
        "Number of Sequences",
        "Mean + Std",
        "Measurements Bounds",
        "# Measurements Per Patient Mean + Std"
    ]
    lab_stats["Number of Sequences"] = lab_stats["n_sequences"].astype(pd.Int64Dtype()).astype(str)
    lab_stats["Mean + Std"] = lab_stats.apply(pretty_mean, axis=1)
    lab_stats["Measurements Bounds"] = lab_stats.apply(pretty_bounds, axis=1)
    lab_stats["# Measurements Per Patient Mean + Std"] = lab_stats.apply(pretty_count_mean, axis=1)
    columns_for_pivot = display_cols
    pretty_df = lab_stats.pivot(index="test_name", columns="source", values=columns_for_pivot)

    # Ensure columns multiindex are sorted per display_cols, per all sources
    all_sources = sorted(lab_stats['source'].dropna().unique())
    pretty_df = pretty_df.reindex(
        columns=pd.MultiIndex.from_product([display_cols, all_sources]),
        fill_value=np.nan
    )
    pretty_df = pretty_df.sort_index(axis=1, level=0)[display_cols]

    # Return pretty
    return pretty_df

pretty_df = get_pretty_lab_stats_df()
pretty_df.to_csv('../data/processed/pretty_combined_lab_stats_by_test_v2.csv', index=False)
display(pretty_df)


Number of Sequences                Mean + Std                   \
                      EHRSHOT MIMIC-IV          EHRSHOT         MIMIC-IV   
test_name                                                                  
A1C                      1745    39996      6.35 ± 1.20      6.55 ± 1.31   
ALB                      4282    55220      3.29 ± 0.80      3.76 ± 0.77   
ALP                      4206    78930   112.49 ± 58.83   107.00 ± 58.35   
ALT                      4219    92599    37.78 ± 25.21    30.17 ± 21.91   
AST                      4220    90639    31.86 ± 19.49    32.74 ± 21.44   
BUN                      5180   153329    23.05 ± 15.22    22.23 ± 14.28   
CA                       5162   121478      8.76 ± 0.71      8.83 ± 0.74   
CL                       5172   148304    102.42 ± 4.98    101.82 ± 5.21   
CO2                      5187   145386     25.93 ± 3.85     25.61 ± 4.10   
CRE                      5161   156044      1.12 ± 0.55      1.08 ± 0.55   
CRP                       725    18147      2.77 ± 3.78    13.55 ± 20.31   
DBIL                     1429     6635      0.34 ± 0.32      0.69 ± 0.89   
GGT                       195     2080  107.00 ± 119.82  112.96 ± 122.38   
GLU                      5150   147195   124.87 ± 39.91   118.29 ± 40.36   
HCT                      5470   164903     32.20 ± 6.76     33.59 ± 6.80   
HDL                      1549    49011    53.60 ± 17.48    55.80 ± 17.69   
HGB                      5369   162168     10.72 ± 2.28     10.97 ± 2.34   
K                        5236   150333      4.14 ± 0.54      4.17 ± 0.54   
LDH                        29    30293   214.28 ± 70.95  269.49 ± 129.62   
LDL                       821    48688    94.90 ± 34.92   104.25 ± 37.77   
MCH                      5294   161671     30.42 ± 2.65     29.71 ± 2.79   
MCHC                     5317   161806     33.24 ± 1.35     32.57 ± 1.61   
MCV                      5304   161661     91.51 ± 6.91     91.21 ± 7.26   
MPV                       371      NaN      9.62 ± 1.62              NaN   
NA                       5193   149303    137.44 ± 4.22    138.74 ± 4.18   
PLT                      5320   162308  197.98 ± 111.37  229.35 ± 113.26   
PT                       3250    78129     18.19 ± 5.73     17.32 ± 6.94   
RBC                      5312   161825      3.55 ± 0.79      3.72 ± 0.81   
RDW                      5315   161640     16.14 ± 2.86     15.14 ± 2.38   
TBIL                     4169    73002      0.64 ± 0.42      0.67 ± 0.56   
TC                       1635    49762   173.52 ± 45.03   187.50 ± 44.65   
TGL                         4    48445  165.70 ± 100.76   140.36 ± 77.55   
TP                       4213    18673      6.60 ± 0.98      6.99 ± 0.72   
WBC                      5345   164291      7.80 ± 4.62      7.71 ± 3.96   

          Measurements Bounds                    \
                      EHRSHOT          MIMIC-IV   
test_name                                         
A1C             [3.60, 10.70]     [3.20, 11.60]   
ALB              [0.50, 6.80]      [0.20, 7.30]   
ALP           [10.00, 345.00]    [1.00, 340.00]   
ALT            [3.00, 141.00]    [1.00, 120.00]   
AST            [3.00, 111.00]    [1.00, 119.00]   
BUN             [1.00, 80.00]     [1.00, 77.00]   
CA              [6.50, 11.10]     [5.70, 12.00]   
CL            [87.00, 118.00]   [81.00, 123.00]   
CO2            [14.00, 38.00]     [8.00, 43.00]   
CRE              [0.06, 3.31]      [0.07, 3.45]   
CRP             [0.10, 17.00]     [0.10, 95.60]   
DBIL             [0.05, 1.79]      [0.10, 4.50]   
GGT            [5.00, 598.00]    [1.00, 609.00]   
GLU            [5.00, 281.00]    [2.00, 279.00]   
HCT            [11.80, 52.70]     [2.00, 69.80]   
HDL            [1.00, 112.00]    [1.00, 135.00]   
HGB             [3.90, 17.60]     [1.70, 22.80]   
K                [2.40, 5.90]      [1.70, 6.60]   
LDH           [92.00, 445.00]   [13.00, 793.00]   
LDL            [6.00, 208.00]   [-4.00, 280.00]   
MCH            [21.70, 3

In [12]:
path = '../data/processed/sequences/MIMIC-IV_sequences.pkl'
with open(path, 'rb') as f:
    mimic_data = pickle.load(f)

path = '../data/processed/sequences/EHRSHOT_sequences.pkl'
with open(path, 'rb') as f:
    ehrshot_data = pickle.load(f)


In [13]:
for seq in mimic_data:
    times = seq['t']
    # if only 1 unique time, print the sequence
    if len(set(times)) < 3:
        print(seq['pid'], seq['test_name'], seq['x'], seq['t'])


10761345 WBC [ 8.8  8.8  1.  11.4 11.6] [  0.        0.        0.      237.95139 237.95139]
10938005 WBC [ 5.1  5.1 31.   5.9  5.9] [ 0.       0.       0.      85.89931 85.89931]
11964105 WBC [ 7.   7.   8.4  8.4 11. ] [ 0.       0.      96.88194 96.88194 96.88194]
12868492 WBC [5.8 5.8 7.  3.8 3.8] [  0.        0.        0.      326.70834 326.70834]


In [14]:
mimic_df = pd.read_csv('../data/processed/mimiciv/MIMIC-IV_processed_df.csv')
mimic_df = mimic_df.sort_values('subject_id').reset_index(drop=True)

dup_mask = mimic_df.duplicated(subset=['subject_id', 'test_name', 'time_delta'], keep=False)
mimic_df = mimic_df.loc[dup_mask].sort_values(['subject_id', 'test_name', 'time_delta'])
display(mimic_df)

,source,subject_id,time,test_name,label,numeric_value,unit,sex,time_delta
395,mimiciv,10000117,2178-04-22 09:15:00,WBC,White Blood Cells,5.4,K/uL,1,1419.01040
399,mimiciv,10000117,2178-04-22 09:15:00,WBC,WBC,1.0,#/hpf,1,1419.01040
1116,mimiciv,10000898,2187-09-26 08:00:00,WBC,WBC,8.0,#/hpf,1,0.00000
1117,mimiciv,10000898,2187-09-26 08:00:00,WBC,White Blood Cells,8.3,K/uL,1,0.00000
1381,mimiciv,10000935,2186-07-30 15:35:00,LDL,"Cholesterol, LDL, Calculated",99.0,mg/dL,1,1340.22570
...,...,...,...,...,...,...,...,...,...
34395031,mimiciv,19999784,2119-12-03 15:00:00,WBC,White Blood Cells,3.9,K/uL,0,162.40277
34395596,mimiciv,19999784,2119-12-25 12:55:00,WBC,White Blood Cells,3.0,K/uL,0,184.31598
34395597,mimiciv,19999784,2119-12-25 12:55:00,WBC,WBC Count,3.0,K/uL,0,184.31598
34394861,mimiciv,19999784,2121-08-18 12:40:00,WBC,WBC Count,3.0,K/uL,0,786.30554


,subject_id,test_name,time_delta,count
0,10000117,WBC,1419.01040,2
1,10000898,WBC,0.00000,2
2,10000935,LDL,1340.22570,2
3,10000935,LDL,1550.94090,2
4,10001725,LDL,0.00000,2
...,...,...,...,...
83236,19999464,WBC,2956.05200,2
83237,19999784,WBC,71.31597,2
83238,19999784,WBC,162.40277,2
83239,19999784,WBC,184.31598,2


In [18]:
mimic_df #.query('test_name == "LDL"')

,source,subject_id,time,test_name,label,numeric_value,unit,sex,time_delta
395,mimiciv,10000117,2178-04-22 09:15:00,WBC,White Blood Cells,5.4,K/uL,1,1419.01040
399,mimiciv,10000117,2178-04-22 09:15:00,WBC,WBC,1.0,#/hpf,1,1419.01040
1116,mimiciv,10000898,2187-09-26 08:00:00,WBC,WBC,8.0,#/hpf,1,0.00000
1117,mimiciv,10000898,2187-09-26 08:00:00,WBC,White Blood Cells,8.3,K/uL,1,0.00000
1381,mimiciv,10000935,2186-07-30 15:35:00,LDL,"Cholesterol, LDL, Calculated",99.0,mg/dL,1,1340.22570
...,...,...,...,...,...,...,...,...,...
34395031,mimiciv,19999784,2119-12-03 15:00:00,WBC,White Blood Cells,3.9,K/uL,0,162.40277
34395596,mimiciv,19999784,2119-12-25 12:55:00,WBC,White Blood Cells,3.0,K/uL,0,184.31598
34395597,mimiciv,19999784,2119-12-25 12:55:00,WBC,WBC Count,3.0,K/uL,0,184.31598
34394861,mimiciv,19999784,2121-08-18 12:40:00,WBC,WBC Count,3.0,K/uL,0,786.30554


In [20]:
mimic_df.groupby(['subject_id', 'test_name', 'time_delta', 'numeric_value']).size().reset_index(name='count').sort_values('count', ascending=False)

,subject_id,test_name,time_delta,numeric_value,count
115207,17852572,WBC,894.240300,5.0,3
101384,16910032,WBC,328.703460,2.0,3
103844,17084065,WBC,1077.989600,3.0,3
120991,18265865,WBC,2016.102800,4.0,3
18543,11320864,WBC,77.385414,6.4,2
...,...,...,...,...,...
52206,13665380,WBC,0.000000,9.1,1
52207,13665396,WBC,4442.271000,1.0,1
52208,13665396,WBC,4442.271000,6.3,1
52213,13666861,WBC,386.038200,1.0,1
